In [1]:
import numpy as np
import pandas as pd

In [2]:
gene_matrix = pd.read_csv('/Users/akshayonly/Main/Scripts/bioinformatics/AMR/data/baumannii/features/gene_matrix.csv')

In [3]:
modelling_data = pd.read_csv('/Users/akshayonly/Main/Scripts/bioinformatics/AMR/data/baumannii/baumannii_modeling_data.csv')

In [4]:
gene_matrix.rename(columns={'sample_id':'Genome ID'}, inplace=True)

In [5]:
modelling_data.head(1)

,Genome ID,Genome Name,Antibiotic,Measurement,Measurement Sign,Measurement Value,Evidence,Resistant Phenotype,Binary Label
0,470.7771,Acinetobacter baumannii strain MRSN7347,imipenem,1,<=,1.0,Laboratory Method,Susceptible,Susceptible


In [6]:
gene_matrix.head(1)

,Genome ID,aac(3)-IId,aac(3)-IIe,aac(3)-Ia,aac(6')-Ian,aac(6')-Ib,aac(6')-Ib',aac(6')-Ib3,aacA16,aadA1,...,sat2,shsP,sul1,sul2,tet(39),tet(A),tet(B),trxLHR,yfdX1,yfdX2
0,470.1359,1,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0


In [7]:
modelling_data['Binary Label'].value_counts()

Binary Label
Susceptible        517
Non-Susceptible    349
Name: count, dtype: int64

In [8]:
# ── Define the three antibiotics ─────────────────────────────────────────────
antibiotics = modelling_data["Antibiotic"].unique().tolist()
print("Antibiotics:", antibiotics)

# ── Build one merged dataframe per antibiotic, then combine ──────────────────
datasets = {}

for antibiotic in antibiotics:

    # Step 1: Filter to this antibiotic
    pheno = modelling_data[modelling_data["Antibiotic"] == antibiotic][
        ["Genome ID", "Binary Label"]
    ].copy()

    # Step 2: Drop missing labels
    pheno = pheno.dropna(subset=["Binary Label"])

    # Step 3: Remove duplicate Genome IDs (keep first)
    pheno = pheno.drop_duplicates(subset="Genome ID")

    # Step 4: Merge with gene_matrix
    merged = pheno.merge(
        gene_matrix.reset_index(),
        on="Genome ID",
        how="inner"
    )

    # Step 5: Reorder columns
    gene_cols = [c for c in merged.columns if c not in ["Genome ID", "Binary Label"]]
    merged = merged[["Genome ID"] + gene_cols + ["Binary Label"]]

    # Step 6: Encode label  →  Susceptible = 0, Non-Susceptible = 1
    merged["Binary Label"] = merged["Binary Label"].map(
        {"Susceptible": 0, "Non-Susceptible": 1}
    )

    datasets[antibiotic] = merged

    print(f"\n── {antibiotic} ──────────────────────────")
    print(f"  Samples : {len(merged)}")
    print(f"  Genes   : {len(gene_cols)}")
    print(f"  Label distribution:\n{merged['Binary Label'].value_counts().to_string()}")

# ── Access individual antibiotic datasets ────────────────────────────────────
# datasets["imipenem"]   →  DataFrame for imipenem
# datasets["meropenem"]  →  DataFrame for meropenem
# datasets["colistin"]   →  DataFrame for colistin   (replace with actual names)

# ── Optional: single stacked dataframe with antibiotic column ────────────────
combined = pd.concat(
    [df.assign(Antibiotic=ab) for ab, df in datasets.items()],
    ignore_index=True
)

# Reorder so Antibiotic sits next to Genome ID
cols = ["Genome ID", "Antibiotic"] + gene_cols + ["Binary Label"]
combined = combined[cols]

combined.head()

Antibiotics: ['imipenem', 'meropenem', 'doripenem']

── imipenem ──────────────────────────
  Samples : 446
  Genes   : 192
  Label distribution:
Binary Label
0    293
1    153

── meropenem ──────────────────────────
  Samples : 205
  Genes   : 192
  Label distribution:
Binary Label
1    109
0     96

── doripenem ──────────────────────────
  Samples : 106
  Genes   : 192
  Label distribution:
Binary Label
0    75
1    31


/var/folders/fy/5h8v4rvx37x6q23ypp1z_m680000gn/T/ipykernel_8756/4288667544.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  gene_matrix.reset_index(),
/var/folders/fy/5h8v4rvx37x6q23ypp1z_m680000gn/T/ipykernel_8756/4288667544.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  gene_matrix.reset_index(),
/var/folders/fy/5h8v4rvx37x6q23ypp1z_m680000gn/T/ipykernel_8756/4288667544.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performan

,Genome ID,Antibiotic,index,aac(3)-IId,aac(3)-IIe,aac(3)-Ia,aac(6')-Ian,aac(6')-Ib,aac(6')-Ib',aac(6')-Ib3,...,shsP,sul1,sul2,tet(39),tet(A),tet(B),trxLHR,yfdX1,yfdX2,Binary Label
0,470.7771,imipenem,345,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,470.8004,imipenem,480,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,470.7715,imipenem,312,0,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
3,470.7505,imipenem,155,0,0,1,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0
4,470.7616,imipenem,243,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (make_scorer, accuracy_score, roc_auc_score,
                             f1_score, recall_score, precision_score,
                             matthews_corrcoef)
import warnings
warnings.filterwarnings("ignore")

# ── Scoring metrics ───────────────────────────────────────────────────────────
scoring = {
    "AUC"       : "roc_auc",
    "F1"        : make_scorer(f1_score),
    "Recall"    : make_scorer(recall_score),
    "Precision" : make_scorer(precision_score),
    "Accuracy"  : make_scorer(accuracy_score),
    "MCC"       : make_scorer(matthews_corrcoef),
}

# ── Models ────────────────────────────────────────────────────────────────────
models = {
    "Logistic Regression" : Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(class_weight="balanced", max_iter=1000,
                                      random_state=42))
    ]),
    "Random Forest" : RandomForestClassifier(
        n_estimators=500, class_weight="balanced",
        random_state=42, n_jobs=-1
    ),
    "Gradient Boosting" : GradientBoostingClassifier(
        n_estimators=200, random_state=42
    ),
    "SVM" : Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    SVC(kernel="rbf", class_weight="balanced",
                       probability=True, random_state=42))
    ]),
}

# ── Cross-validation setup ────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Step 1: Split FIRST, store holdout sets ───────────────────────────────────
holdout_sets = {}

print("── TRAIN / TEST SPLIT ───────────────────────────────────────────────")
for antibiotic, df in datasets.items():

    X = df.drop(columns=["Genome ID", "Binary Label"]).values
    y = df["Binary Label"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        stratify=y,       # preserve class ratio in both splits
        random_state=42
    )

    holdout_sets[antibiotic] = {
        "X_train" : X_train,
        "X_test"  : X_test,
        "y_train" : y_train,
        "y_test"  : y_test,
    }

    print(f"\n{antibiotic.upper()}")
    print(f"  Train : n={len(y_train)}  |  "
          f"Non-Susceptible={y_train.sum()}  "
          f"Susceptible={(y_train == 0).sum()}")
    print(f"  Test  : n={len(y_test)}   |  "
          f"Non-Susceptible={y_test.sum()}  "
          f"Susceptible={(y_test == 0).sum()}")

# ── Step 2: Baseline modelling on TRAINING SET only ──────────────────────────
all_results = []

print("\n\n── BASELINE MODELLING (5-Fold CV on Training Set only) ─────────────")

for antibiotic, splits in holdout_sets.items():

    X_train = splits["X_train"]
    y_train = splits["y_train"]

    print(f"\n{'='*55}")
    print(f"  {antibiotic.upper()}  |  "
          f"Train n={len(y_train)}  |  "
          f"Non-Susceptible={y_train.sum()}  "
          f"Susceptible={(y_train == 0).sum()}")
    print(f"{'='*55}")

    for model_name, model in models.items():

        # CV runs on X_train / y_train — test set is untouched
        cv_results = cross_validate(
            model, X_train, y_train,
            cv=cv,
            scoring=scoring,
            return_train_score=False,
            n_jobs=-1
        )

        row = {"Antibiotic": antibiotic, "Model": model_name}
        for metric in scoring:
            scores = cv_results[f"test_{metric}"]
            row[f"{metric} Mean"] = scores.mean().round(3)
            row[f"{metric} Std"]  = scores.std().round(3)

        all_results.append(row)

        print(f"\n  {model_name}")
        print(f"    AUC       : {row['AUC Mean']:.3f} ± {row['AUC Std']:.3f}")
        print(f"    F1        : {row['F1 Mean']:.3f} ± {row['F1 Std']:.3f}")
        print(f"    Recall    : {row['Recall Mean']:.3f} ± {row['Recall Std']:.3f}")
        print(f"    Precision : {row['Precision Mean']:.3f} ± {row['Precision Std']:.3f}")
        print(f"    MCC       : {row['MCC Mean']:.3f} ± {row['MCC Std']:.3f}")

# ── Summary table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results)

results_df["AUC Rank"] = (
    results_df.groupby("Antibiotic")["AUC Mean"]
    .rank(ascending=False)
    .astype(int)
)

print("\n\n── SUMMARY TABLE (sorted by Antibiotic + AUC Rank) ─────────────────")
summary = results_df[["Antibiotic", "Model", "AUC Mean", "AUC Std",
                       "F1 Mean", "Recall Mean", "MCC Mean", "AUC Rank"]]
print(summary.sort_values(["Antibiotic", "AUC Rank"]).to_string(index=False))

── TRAIN / TEST SPLIT ───────────────────────────────────────────────

IMIPENEM
  Train : n=356  |  Non-Susceptible=122  Susceptible=234
  Test  : n=90   |  Non-Susceptible=31  Susceptible=59

MEROPENEM
  Train : n=164  |  Non-Susceptible=87  Susceptible=77
  Test  : n=41   |  Non-Susceptible=22  Susceptible=19

DORIPENEM
  Train : n=84  |  Non-Susceptible=25  Susceptible=59
  Test  : n=22   |  Non-Susceptible=6  Susceptible=16


── BASELINE MODELLING (5-Fold CV on Training Set only) ─────────────

  IMIPENEM  |  Train n=356  |  Non-Susceptible=122  Susceptible=234

  Logistic Regression
    AUC       : 0.942 ± 0.032
    F1        : 0.862 ± 0.045
    Recall    : 0.869 ± 0.047
    Precision : 0.858 ± 0.058
    MCC       : 0.790 ± 0.070

  Random Forest
    AUC       : 0.959 ± 0.021
    F1        : 0.870 ± 0.045
    Recall    : 0.845 ± 0.028
    Precision : 0.899 ± 0.068
    MCC       : 0.807 ± 0.071

  Gradient Boosting
    AUC       : 0.955 ± 0.020
    F1        : 0.885 ± 0.042
    Rec

In [10]:
from sklearn.model_selection import LeaveOneOut

# ── Replace 5-fold CV with LOOCV for doripenem only ──────────────────────────
loocv = LeaveOneOut()

antibiotic = "doripenem"
splits     = holdout_sets[antibiotic]
X_train    = splits["X_train"]
y_train    = splits["y_train"]

print(f"── DORIPENEM LOOCV (n={len(y_train)}) ──────────────────────────────")

loocv_results = []

for model_name, model in models.items():

    cv_results = cross_validate(
        model, X_train, y_train,
        cv=loocv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    row = {"Antibiotic": antibiotic, "Model": model_name}
    for metric in scoring:
        scores = cv_results[f"test_{metric}"]
        row[f"{metric} Mean"] = scores.mean().round(3)
        row[f"{metric} Std"]  = scores.std().round(3)

    loocv_results.append(row)

    print(f"\n  {model_name}")
    print(f"    AUC       : {row['AUC Mean']:.3f} ± {row['AUC Std']:.3f}")
    print(f"    F1        : {row['F1 Mean']:.3f} ± {row['F1 Std']:.3f}")
    print(f"    Recall    : {row['Recall Mean']:.3f} ± {row['Recall Std']:.3f}")
    print(f"    Precision : {row['Precision Mean']:.3f} ± {row['Precision Std']:.3f}")
    print(f"    MCC       : {row['MCC Mean']:.3f} ± {row['MCC Std']:.3f}")

loocv_df = pd.DataFrame(loocv_results)

── DORIPENEM LOOCV (n=84) ──────────────────────────────

  Logistic Regression
    AUC       : nan ± nan
    F1        : 0.250 ± 0.433
    Recall    : 0.250 ± 0.433
    Precision : 0.250 ± 0.433
    MCC       : 0.000 ± 0.000

  Random Forest
    AUC       : nan ± nan
    F1        : 0.286 ± 0.452
    Recall    : 0.286 ± 0.452
    Precision : 0.286 ± 0.452
    MCC       : 0.000 ± 0.000

  Gradient Boosting
    AUC       : nan ± nan
    F1        : 0.298 ± 0.457
    Recall    : 0.298 ± 0.457
    Precision : 0.298 ± 0.457
    MCC       : 0.000 ± 0.000

  SVM
    AUC       : nan ± nan
    F1        : 0.250 ± 0.433
    Recall    : 0.250 ± 0.433
    Precision : 0.250 ± 0.433
    MCC       : 0.000 ± 0.000


In [11]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Reduced-complexity GBM variants for doripenem ────────────────────────────
gbm_variants = {
    "GBM Default (baseline)"   : GradientBoostingClassifier(
        n_estimators=200,
        random_state=42
    ),
    "GBM Fewer Trees"          : GradientBoostingClassifier(
        n_estimators=50,
        random_state=42
    ),
    "GBM Shallow Trees"        : GradientBoostingClassifier(
        n_estimators=100,
        max_depth=2,           # default is 3; shallower = less complex
        random_state=42
    ),
    "GBM Strong Regularisation": GradientBoostingClassifier(
        n_estimators=100,
        max_depth=2,
        min_samples_leaf=5,    # node must have ≥5 samples — prevents tiny splits
        min_samples_split=10,  # split only if ≥10 samples in node
        max_features=0.5,      # use only 50% of features per split
        random_state=42
    ),
    "GBM High Shrinkage"       : GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.01,    # default is 0.1; slower learning = less overfit
        max_depth=2,
        random_state=42
    ),
    "GBM Conservative"         : GradientBoostingClassifier(
        n_estimators=50,
        learning_rate=0.05,
        max_depth=2,
        min_samples_leaf=5,
        min_samples_split=10,
        max_features=0.5,
        subsample=0.8,         # use 80% of samples per tree — stochastic GBM
        random_state=42
    ),
}

# ── Run on doripenem training set only ───────────────────────────────────────
antibiotic = "doripenem"
X_train    = holdout_sets[antibiotic]["X_train"]
y_train    = holdout_sets[antibiotic]["y_train"]

print(f"── GBM COMPLEXITY REDUCTION  |  doripenem  |  Train n={len(y_train)} ──")
print(f"   Non-Susceptible={y_train.sum()}  Susceptible={(y_train==0).sum()}\n")

gbm_results = []

for variant_name, model in gbm_variants.items():

    cv_results = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    row = {"Variant": variant_name}
    for metric in scoring:
        scores = cv_results[f"test_{metric}"]
        row[f"{metric} Mean"] = scores.mean().round(3)
        row[f"{metric} Std"]  = scores.std().round(3)

    gbm_results.append(row)

    print(f"  {variant_name}")
    print(f"    AUC       : {row['AUC Mean']:.3f} ± {row['AUC Std']:.3f}")
    print(f"    F1        : {row['F1 Mean']:.3f} ± {row['F1 Std']:.3f}")
    print(f"    Recall    : {row['Recall Mean']:.3f} ± {row['Recall Std']:.3f}")
    print(f"    Precision : {row['Precision Mean']:.3f} ± {row['Precision Std']:.3f}")
    print(f"    MCC       : {row['MCC Mean']:.3f} ± {row['MCC Std']:.3f}\n")

# ── Summary ───────────────────────────────────────────────────────────────────
gbm_df = pd.DataFrame(gbm_results)
gbm_df["AUC Rank"] = gbm_df["AUC Mean"].rank(ascending=False).astype(int)

print("── SUMMARY ──────────────────────────────────────────────────────────")
print(gbm_df[["Variant", "AUC Mean", "AUC Std", "F1 Mean",
              "Recall Mean", "MCC Mean", "AUC Rank"]]
      .sort_values("AUC Rank")
      .to_string(index=False))

── GBM COMPLEXITY REDUCTION  |  doripenem  |  Train n=84 ──
   Non-Susceptible=25  Susceptible=59

  GBM Default (baseline)
    AUC       : 1.000 ± 0.000
    F1        : 1.000 ± 0.000
    Recall    : 1.000 ± 0.000
    Precision : 1.000 ± 0.000
    MCC       : 1.000 ± 0.000

  GBM Fewer Trees
    AUC       : 1.000 ± 0.000
    F1        : 1.000 ± 0.000
    Recall    : 1.000 ± 0.000
    Precision : 1.000 ± 0.000
    MCC       : 1.000 ± 0.000

  GBM Shallow Trees
    AUC       : 1.000 ± 0.000
    F1        : 1.000 ± 0.000
    Recall    : 1.000 ± 0.000
    Precision : 1.000 ± 0.000
    MCC       : 1.000 ± 0.000

  GBM Strong Regularisation
    AUC       : 1.000 ± 0.000
    F1        : 1.000 ± 0.000
    Recall    : 1.000 ± 0.000
    Precision : 1.000 ± 0.000
    MCC       : 1.000 ± 0.000

  GBM High Shrinkage
    AUC       : 1.000 ± 0.000
    F1        : 1.000 ± 0.000
    Recall    : 1.000 ± 0.000
    Precision : 1.000 ± 0.000
    MCC       : 1.000 ± 0.000

  GBM Conservative
    AUC       :

In [12]:
# ── Drop doripenem ────────────────────────────────────────────────────────────
datasets    = {k: v for k, v in datasets.items()    if k != "doripenem"}
holdout_sets = {k: v for k, v in holdout_sets.items() if k != "doripenem"}

print("Remaining antibiotics:", list(datasets.keys()))
print("Holdout sets         :", list(holdout_sets.keys()))

# ── Confirm sizes ─────────────────────────────────────────────────────────────
for antibiotic, splits in holdout_sets.items():
    y_train = splits["y_train"]
    y_test  = splits["y_test"]
    print(f"\n{antibiotic.upper()}")
    print(f"  Train : n={len(y_train)}  |  Non-Susceptible={y_train.sum()}  Susceptible={(y_train==0).sum()}")
    print(f"  Test  : n={len(y_test)}   |  Non-Susceptible={y_test.sum()}  Susceptible={(y_test==0).sum()}")

Remaining antibiotics: ['imipenem', 'meropenem']
Holdout sets         : ['imipenem', 'meropenem']

IMIPENEM
  Train : n=356  |  Non-Susceptible=122  Susceptible=234
  Test  : n=90   |  Non-Susceptible=31  Susceptible=59

MEROPENEM
  Train : n=164  |  Non-Susceptible=87  Susceptible=77
  Test  : n=41   |  Non-Susceptible=22  Susceptible=19


In [13]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import (
    SelectKBest, chi2,
    RFE, SelectFromModel
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (make_scorer, roc_auc_score, f1_score,
                             recall_score, matthews_corrcoef)
import warnings
warnings.filterwarnings("ignore")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "AUC"    : "roc_auc",
    "F1"     : make_scorer(f1_score),
    "Recall" : make_scorer(recall_score),
    "MCC"    : make_scorer(matthews_corrcoef),
}

# ── Three classes of feature selection ───────────────────────────────────────
# 1. FILTER    — chi2 statistical test, no model involved, fast
# 2. EMBEDDED  — model ranks features intrinsically during training (RF importance)
# 3. WRAPPER   — RFE iteratively removes weakest features using a model

def evaluate(name, pipeline, X_train, y_train):
    """Run 5-fold CV and return result row."""
    res = cross_validate(pipeline, X_train, y_train,
                         cv=cv, scoring=scoring,
                         return_train_score=False, n_jobs=-1)
    return {
        "Method"      : name,
        "AUC Mean"    : res["test_AUC"].mean().round(3),
        "AUC Std"     : res["test_AUC"].std().round(3),
        "F1 Mean"     : res["test_F1"].mean().round(3),
        "Recall Mean" : res["test_Recall"].mean().round(3),
        "MCC Mean"    : res["test_MCC"].mean().round(3),
    }


# ── Run feature selection for each antibiotic ─────────────────────────────────
for antibiotic, splits in holdout_sets.items():

    X_train = splits["X_train"]
    y_train = splits["y_train"]
    n_features = X_train.shape[1]   # 192 genes

    print(f"\n{'='*60}")
    print(f"  {antibiotic.upper()}  |  Train n={len(y_train)}  |  Features={n_features}")
    print(f"{'='*60}")

    results = []

    # ── Baseline: all 192 genes, no selection ────────────────────────────────
    baseline = Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    GradientBoostingClassifier(n_estimators=200, random_state=42))
    ])
    results.append(evaluate("Baseline (all 192 genes)", baseline, X_train, y_train))

    # ── 1. FILTER: Chi-Square (top k genes) ──────────────────────────────────
    # Chi2 measures statistical dependence between each gene and the label.
    # Binary features (0/1) satisfy chi2's non-negative input requirement.
    for k in [50, 100, 150]:
        pipe = Pipeline([
            ("select", SelectKBest(chi2, k=k)),
            ("scaler", StandardScaler()),
            ("clf",    GradientBoostingClassifier(n_estimators=200, random_state=42))
        ])
        results.append(evaluate(f"Filter: Chi2 (top {k})", pipe, X_train, y_train))

    # ── 2. EMBEDDED: Random Forest importance threshold ───────────────────────
    # RF trains once and ranks all 192 genes by Gini importance.
    # SelectFromModel keeps genes above a mean importance threshold.
    for threshold in ["mean", "median"]:
        pipe = Pipeline([
            ("select", SelectFromModel(
                RandomForestClassifier(n_estimators=500, class_weight="balanced",
                                       random_state=42, n_jobs=-1),
                threshold=threshold
            )),
            ("scaler", StandardScaler()),
            ("clf",    GradientBoostingClassifier(n_estimators=200, random_state=42))
        ])
        results.append(evaluate(f"Embedded: RF importance (>{threshold})",
                                pipe, X_train, y_train))

    # ── 3. WRAPPER: RFE with Logistic Regression ──────────────────────────────
    # RFE iteratively removes the weakest feature (lowest LR coefficient)
    # until the target number remains. Most expensive but most thorough.
    for k in [50, 100]:
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("select", RFE(
                estimator=LogisticRegression(class_weight="balanced",
                                             max_iter=1000, random_state=42),
                n_features_to_select=k,
                step=10           # remove 10 genes per iteration for speed
            )),
            ("clf",    GradientBoostingClassifier(n_estimators=200, random_state=42))
        ])
        results.append(evaluate(f"Wrapper: RFE-LR (top {k})", pipe, X_train, y_train))

    # ── Summary ───────────────────────────────────────────────────────────────
    res_df = pd.DataFrame(results)
    res_df["AUC Rank"] = res_df["AUC Mean"].rank(ascending=False).astype(int)

    print(res_df[["Method", "AUC Mean", "AUC Std", "F1 Mean",
                  "Recall Mean", "MCC Mean", "AUC Rank"]]
          .sort_values("AUC Rank")
          .to_string(index=False))


  IMIPENEM  |  Train n=356  |  Features=192
                           Method  AUC Mean  AUC Std  F1 Mean  Recall Mean  MCC Mean  AUC Rank
           Filter: Chi2 (top 100)     0.963    0.021    0.890        0.878     0.833         1
         Wrapper: RFE-LR (top 50)     0.961    0.024    0.881        0.861     0.822         2
  Embedded: RF importance (>mean)     0.960    0.013    0.880        0.853     0.822         3
            Filter: Chi2 (top 50)     0.957    0.019    0.885        0.869     0.828         4
         Baseline (all 192 genes)     0.955    0.020    0.885        0.869     0.827         5
        Wrapper: RFE-LR (top 100)     0.955    0.019    0.872        0.853     0.809         5
           Filter: Chi2 (top 150)     0.954    0.024    0.881        0.869     0.821         7
Embedded: RF importance (>median)     0.953    0.023    0.886        0.878     0.827         8

  MEROPENEM  |  Train n=164  |  Features=192
                           Method  AUC Mean  AUC Std  